<a href="https://colab.research.google.com/github/shims79757-lang/Elevance-Skills-Projects/blob/main/Interactive_App_Category_Heatmap_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ==========================================
# 1. LOAD DATASETS & INITIAL PREPROCESSING
# ==========================================
apps = pd.read_csv(
    "https://raw.githubusercontent.com/shims79757-lang/Elevance-Skills-Projects/main/googleplaystore.csv"
)
reviews = pd.read_csv(
    "https://raw.githubusercontent.com/shims79757-lang/Elevance-Skills-Projects/main/googleplaystore_user_reviews.csv"
)

data = apps.copy()

# Numeric conversions
data["Rating"] = pd.to_numeric(data["Rating"], errors="coerce")
data["Reviews"] = pd.to_numeric(data["Reviews"], errors="coerce")

# Clean and convert Installs
data["Installs"] = (
    data["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
data["Installs"] = pd.to_numeric(data["Installs"], errors="coerce")


# Convert Size to MB
def convert_size(size):
  size = str(size).strip()
  if size.endswith("M"):
    return float(size[:-1])
  elif size.endswith("k"):
    return float(size[:-1]) / 1024
  return np.nan


data["Size_MB"] = data["Size"].apply(convert_size)

# Merge Sentiment Subjectivity from reviews
app_subjectivity = (
    reviews.groupby("App")["Sentiment_Subjectivity"].mean().reset_index()
)
data = data.merge(app_subjectivity, on="App", how="left")

# Parse Last Updated date
data["Last Updated"] = pd.to_datetime(data["Last Updated"], errors="coerce")

# ==========================================
# 2. FILTERING & TRANSLATIONS
# ==========================================
# 1. Categories beginning with 'E', 'C', or 'B'
# 2. Rating >= 4.0, Installs > 10,000, Reviews > 500
# 3. Size between 15 MB and 80 MB (inclusive)
# 4. Sentiment subjectivity > 0.5
# 5. Exclude app names starting with 'X', 'Y', 'Z'
# 6. Exclude app names containing the letter 'S' (case-insensitive)
data = data[
    data["Category"].astype(str).str.upper().str.startswith(("E", "C", "B"))
    & (data["Rating"] >= 4.0)
    & (data["Installs"] > 10000)
    & (data["Reviews"] > 500)
    & (data["Size_MB"] >= 15)
    & (data["Size_MB"] <= 80)
    & (data["Sentiment_Subjectivity"] > 0.5)
    & (~data["App"].astype(str).str.upper().str.startswith(("X", "Y", "Z")))
    & (~data["App"].astype(str).str.contains("s", case=False, na=False))
].copy()

# Deduplicate
data = data.drop_duplicates(subset=["App", "Category"]).reset_index(drop=True)

# Required category translations
translations = {
    "BEAUTY": "सुंदरता",  # Hindi
    "BUSINESS": "வணிகம்",  # Tamil
    "DATING": "Partnersuche",  # German
    "Beauty": "सुंदरता",
    "Business": "வணிகம்",
    "Dating": "Partnersuche",
}
data["Display_Category"] = data["Category"].replace(translations)

# Select top 5 eligible categories by total installs
top_5_categories = (
    data.groupby("Category")["Installs"].sum().nlargest(5).index.tolist()
)
data = data[data["Category"].isin(top_5_categories)].copy()

# ==========================================
# 3. MONTHLY METRICS & 3-MONTH MA FORECAST
# ==========================================
month_names = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]

# Process monthly timelines and forecasts for each category
category_matrices = {}
all_years = set()

for cat in top_5_categories:
  cat_df = (
      data[data["Category"] == cat].dropna(subset=["Last Updated"]).copy()
  )
  cat_df["YearMonth"] = cat_df["Last Updated"].dt.to_period("M")

  # Aggregate historical monthly totals
  monthly = (
      cat_df.groupby("YearMonth")
      .agg(Installs=("Installs", "sum"), Reviews=("Reviews", "sum"))
      .reset_index()
  )

  if monthly.empty:
    continue

  # Build continuous monthly sequence
  full_range = pd.period_range(
      start=monthly["YearMonth"].min(),
      end=monthly["YearMonth"].max(),
      freq="M",
  )
  monthly = (
      monthly.set_index("YearMonth")
      .reindex(full_range)
      .fillna(0)
      .reset_index()
  )
  monthly.rename(columns={"index": "YearMonth"}, inplace=True)

  # MoM Growth & 3-Month Rolling Average
  monthly["Growth_Pct"] = monthly["Installs"].pct_change() * 100
  monthly["Rolling_Avg"] = monthly["Installs"].rolling(window=3).mean()
  monthly["Status"] = "Actual"

  # Generate 3-Month Moving-Average Forecast
  last_month = monthly["YearMonth"].iloc[-1]
  last_3_installs = list(monthly["Installs"].iloc[-3:])
  forecast_rows = []

  for step in range(1, 4):
    fcst_month = last_month + step
    fcst_val = float(np.mean(last_3_installs[-3:]))
    prev_val = (
        forecast_rows[-1]["Installs"] if forecast_rows else last_3_installs[-1]
    )
    growth_val = (
        ((fcst_val - prev_val) / prev_val * 100) if prev_val > 0 else 0.0
    )

    combined = last_3_installs + [fcst_val]
    r_avg = float(np.mean(combined[-3:]))

    forecast_rows.append({
        "YearMonth": fcst_month,
        "Installs": fcst_val,
        "Reviews": 0,
        "Growth_Pct": growth_val,
        "Rolling_Avg": r_avg,
        "Status": "Forecast",
    })
    last_3_installs.append(fcst_val)

  full_series = pd.concat(
      [monthly, pd.DataFrame(forecast_rows)], ignore_index=True
  )
  full_series["Year"] = full_series["YearMonth"].dt.year
  full_series["Month"] = full_series["YearMonth"].dt.month

  category_matrices[cat] = full_series
  all_years.update(full_series["Year"].tolist())

years_sorted = sorted(list(all_years))

# ==========================================
# 4. ASSEMBLE CALENDAR HEATMAP & TRACES
# ==========================================
fig = go.Figure()
traces_per_category = 3  # Heatmap, High Growth markers, Forecast markers
buttons = []

for cat_idx, cat in enumerate(top_5_categories):
  if cat not in category_matrices:
    continue

  df_cat = category_matrices[cat]
  display_name = translations.get(cat, cat)

  # Initialize grid matrices
  z_matrix = []
  customdata_matrix = []
  text_matrix = []

  high_growth_x, high_growth_y, high_growth_text = [], [], []
  forecast_x, forecast_y, forecast_text = [], [], []

  for yr in years_sorted:
    z_row = []
    cd_row = []
    txt_row = []

    for m_num in range(1, 13):
      m_str = month_names[m_num - 1]
      match = df_cat[(df_cat["Year"] == yr) & (df_cat["Month"] == m_num)]

      if not match.empty:
        row = match.iloc[0]
        installs = row["Installs"]
        growth = row["Growth_Pct"]
        rolling = row["Rolling_Avg"]
        reviews_cnt = row["Reviews"]
        status = row["Status"]

        z_row.append(installs)

        growth_str = f"{growth:+.1f}%" if pd.notnull(growth) else "N/A"
        rolling_str = f"{rolling:,.0f}" if pd.notnull(rolling) else "N/A"
        reviews_str = (
            f"{reviews_cnt:,.0f}" if status == "Actual" else "Projected (N/A)"
        )

        cd_row.append([
            f"{m_str} {yr}",
            growth_str,
            rolling_str,
            reviews_str,
            status,
        ])

        if status == "Forecast":
          txt_row.append(f"[Forecast]<br>{installs:,.0f}")
          forecast_x.append(m_str)
          forecast_y.append(yr)
          forecast_text.append(f"{installs:,.0f}")
        elif pd.notnull(growth) and growth > 20.0:
          txt_row.append(f"▲ +{growth:.0f}%<br>{installs:,.0f}")
          high_growth_x.append(m_str)
          high_growth_y.append(yr)
          high_growth_text.append(f"+{growth:.1f}%")
        else:
          txt_row.append(f"{installs:,.0f}" if installs > 0 else "")
      else:
        z_row.append(np.nan)
        cd_row.append(["", "N/A", "N/A", "N/A", "No Data"])
        txt_row.append("")

    z_matrix.append(z_row)
    customdata_matrix.append(cd_row)
    text_matrix.append(txt_row)

  is_initial = cat_idx == 0

  # Trace 1: Base Heatmap
  fig.add_trace(
      go.Heatmap(
          z=z_matrix,
          x=month_names,
          y=years_sorted,
          customdata=customdata_matrix,
          text=text_matrix,
          texttemplate="%{text}",
          colorscale="Blues",
          showscale=True,
          colorbar=dict(
              title="Installs",
              thickness=15,
              len=0.8,
          ),
          hovertemplate=(
              "<b>%{customdata[0]}</b><br>"
              "Status: <b>%{customdata[4]}</b><br>"
              "Monthly Installs: %{z:,.0f}<br>"
              "MoM Growth: %{customdata[1]}<br>"
              "3-Month Rolling Avg: %{customdata[2]}<br>"
              "Review Count: %{customdata[3]}"
              "<extra></extra>"
          ),
          visible=is_initial,
          name=f"{display_name} Heatmap",
      )
  )

  # Trace 2: High Growth (>20%) Overlay Markers
  fig.add_trace(
      go.Scatter(
          x=high_growth_x,
          y=high_growth_y,
          mode="markers",
          marker=dict(
              symbol="triangle-up",
              size=14,
              color="#00C853",
              line=dict(color="white", width=1.5),
          ),
          name="Growth > 20%",
          hoverinfo="skip",
          visible=is_initial,
          showlegend=True,
      )
  )

  # Trace 3: Forecast Distinct Visual Overlay Markers
  fig.add_trace(
      go.Scatter(
          x=forecast_x,
          y=forecast_y,
          mode="markers",
          marker=dict(
              symbol="diamond-open",
              size=18,
              color="#FF6D00",
              line=dict(color="#FF6D00", width=2.5),
          ),
          name="3-Month MA Forecast",
          hoverinfo="skip",
          visible=is_initial,
          showlegend=True,
      )
  )

# ==========================================
# 5. DYNAMIC CATEGORY SELECTOR (DROPDOWN)
# ==========================================
total_traces = len(top_5_categories) * traces_per_category

for idx, cat in enumerate(top_5_categories):
  display_name = translations.get(cat, cat)
  visibility = [False] * total_traces

  # Enable the 3 traces for this category
  for t in range(traces_per_category):
    visibility[idx * traces_per_category + t] = True

  buttons.append(
      dict(
          label=display_name,
          method="update",
          args=[
              {"visible": visibility},
              {
                  "title.text": (
                      f"<b>Monthly Installs Calendar Heatmap —"
                      f" {display_name}</b><br><sup>MoM Growth (>20%"
                      " highlighted) & 3-Month Moving-Average Forecast</sup>"
                  )
              },
          ],
      )
  )

first_cat_display = translations.get(top_5_categories[0], top_5_categories[0])

fig.update_layout(
    title=dict(
        text=(
            f"<b>Monthly Installs Calendar Heatmap — {first_cat_display}</b><br>"
            "<sup>MoM Growth (>20% highlighted) & 3-Month Moving-Average"
            " Forecast</sup>"
        ),
        x=0.5,
    ),
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.18,
            yanchor="top",
            bgcolor="#ffffff",
            bordercolor="#c8d4e3",
            borderwidth=1,
        )
    ],
    xaxis=dict(
        title="Month",
        tickmode="array",
        tickvals=month_names,
        gridcolor="#f0f0f0",
    ),
    yaxis=dict(
        title="Year",
        type="category",
        dtick=1,
        autorange="reversed",  # Earliest years at top, recent at bottom
        gridcolor="#f0f0f0",
    ),
    template="plotly_white",
    height=650,
    margin=dict(l=70, r=60, t=130, b=70),
    legend=dict(orientation="h", y=1.06, x=1.0, xanchor="right"),
)

# ==========================================
# 6. TIME GATING (6:00 PM – 9:00 PM IST)
# ==========================================
current_time = datetime.now(ZoneInfo("Asia/Kolkata"))
print("Current IST:", current_time.strftime("%d %B %Y, %I:%M %p"))

if 18 <= current_time.hour < 21:
  fig.show()
else:
  print(
      "Visualization unavailable.\n"
      "This graph can only be viewed between 6:00 PM and 9:00 PM IST."
  )

Current IST: 24 September 2026, 07:07 PM
